# Anime Action Scene — 2D Baseline Inference
**Moore-AnimateAnyone + Anything V5**

순서대로 셀 실행하면 됩니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 또는 A100 GPU** 선택 후 시작  
> ⚠️ 셀 2(의존성 설치) 실행 후 런타임이 자동 재시작됩니다. **재시작 후 셀 1부터 다시 실행하세요.**

In [1]:
# ── 0. GPU 확인 ───────────────────────────────────────────────────────────────
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

GPU : Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [2]:
# ── 1. 레포 클론 ───────────────────────────────────────────────────────────────
import os

if not os.path.exists('/content/anime-action-scene'):
    !git clone https://github.com/goldkangsan/anime-action-scene.git /content/anime-action-scene

%cd /content/anime-action-scene
!ls

/content/anime-action-scene
champ			     configs	       run.sh
colab_2d_baseline.ipynb      README_KR.md      scripts
colab_animate_anyone.ipynb   requirements.txt  src
colab_champ_inference.ipynb  run_gradio.py     tools


In [ ]:
# ── 2. 의존성 설치 (처음 한 번만 실행) ────────────────────────────────────────
# ⚠️ 실행 후 런타임 자동 재시작 → 재시작 후 셀 0, 1 실행하고 셀 3으로 바로!

!pip install -q --upgrade pip

# 검증된 버전 조합 (실제 테스트 완료)
# huggingface_hub 0.23.0+ 에서 cached_download 삭제 → <0.23.0 고정 필수
!pip install -q \
    "diffusers==0.24.0" \
    "transformers>=4.36.0,<4.40.0" \
    "huggingface_hub>=0.19.0,<0.23.0" \
    "accelerate"

!pip install -q \
    omegaconf \
    einops \
    "controlnet-aux" \
    "onnxruntime-gpu" \
    imageio \
    "imageio[ffmpeg]" \
    av \
    gradio

print('\n✅ 설치 완료 — 런타임 자동 재시작 중...')
import os; os.kill(os.getpid(), 9)

In [ ]:
# ── 3. import 테스트 ───────────────────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings('ignore')

os.chdir('/content/anime-action-scene')
sys.path.insert(0, '/content/anime-action-scene')

# 버전 확인
import diffusers, transformers, huggingface_hub
print(f'diffusers      : {diffusers.__version__}')    # 0.24.0
print(f'transformers   : {transformers.__version__}')  # 4.36~4.39
print(f'huggingface_hub: {huggingface_hub.__version__}')  # <0.23.0

from src.dwpose import DWposeDetector
from src.models.pose_guider import PoseGuider
from src.models.unet_2d_condition import UNet2DConditionModel
from src.models.unet_3d import UNet3DConditionModel
from src.pipelines.pipeline_pose2vid_long import Pose2VideoPipeline
from src.utils.util import get_fps, read_frames, save_videos_grid

print('\n✅ 모든 import 성공!')

In [ ]:
# ── 4. 가중치 다운로드 (~4GB, 처음 한 번만) ───────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

# SD 1.5 기반 기본 가중치
!python tools/download_weights.py

# Anything V5 추가 (애니 스타일용)
!python tools/download_weights.py --anime

In [ ]:
# ── 5. 참조 이미지 업로드 ─────────────────────────────────────────────────────
import os, shutil
from google.colab import files
from pathlib import Path

os.chdir('/content/anime-action-scene')
Path('inputs').mkdir(exist_ok=True)

print('📁 애니 캐릭터 이미지를 업로드하세요 (jpg/png)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/ref.png')
    print(f'  → inputs/ref.png 저장 완료')

In [ ]:
# ── 6. 액션 영상 업로드 ───────────────────────────────────────────────────────
import os, shutil
from google.colab import files
from pathlib import Path

os.chdir('/content/anime-action-scene')
Path('inputs').mkdir(exist_ok=True)

print('🎬 액션 영상을 업로드하세요 (mp4)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/driving.mp4')
    print(f'  → inputs/driving.mp4 저장 완료')

In [ ]:
# ── 7. 업로드 확인 ─────────────────────────────────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

from PIL import Image
import cv2
from IPython.display import display

ref = Image.open('inputs/ref.png')
print(f'참조 이미지: {ref.size}')
display(ref.resize((200, int(200 * ref.height / ref.width))))

cap = cv2.VideoCapture('inputs/driving.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'드라이빙 영상: {total}프레임 @ {fps:.1f}fps')

In [ ]:
# ── 8. DWPose 추출 (영상 → 포즈 비디오) ───────────────────────────────────────
import os
os.chdir('/content/anime-action-scene')
from pathlib import Path
Path('outputs/pose').mkdir(parents=True, exist_ok=True)

!python tools/vid2pose.py \
    --video_path inputs/driving.mp4 \
    --output_path outputs/pose/driving_kps.mp4 \
    --device cuda \
    --max_frames 32

In [ ]:
# ── 9-A. Inference — SD 1.5 (비교용 baseline) ─────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

!python scripts/pose2vid.py \
    --config ./configs/prompts/animation.yaml \
    -W 384 -H 512 -L 32 \
    --steps 20 \
    --cfg 3.5 \
    --seed 42 \
    --device cuda

In [ ]:
# ── 9-B. Inference — Anything V5 (애니 특화) ──────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

!python scripts/pose2vid.py \
    --config ./configs/prompts/animation_anime.yaml \
    -W 384 -H 512 -L 32 \
    --steps 20 \
    --cfg 3.5 \
    --seed 42 \
    --device cuda

In [ ]:
# ── 10. 결과 확인 ──────────────────────────────────────────────────────────────
import os, glob
os.chdir('/content/anime-action-scene')
from IPython.display import Video, display

results = sorted(glob.glob('output/**/*.mp4', recursive=True))
print(f'생성된 영상 {len(results)}개:')
for r in results:
    print(f'  {r}')

if results:
    print('\n▶ 최신 결과 미리보기:')
    display(Video(results[-1], width=512, embed=True))

In [ ]:
# ── 11. 결과 다운로드 ──────────────────────────────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

!zip -r results.zip output/
from google.colab import files
files.download('results.zip')
print('✅ results.zip 다운로드 시작')